# Predicting Viral AI Tweets — SWA2124 Social & Web Analytics

**Group:** Jehuda Rhema Chang (leader), Brandon Wong Kai Ian, Cheah Choon Keat, Nihal, Palani

This notebook is fully reproducible in Google Colab. It:
1. downloads the open **tweets_ai** dataset directly from **Harvard Dataverse** (DOI `10.7910/DVN/NHLEJL`),
2. preprocesses the tweets and engineers features (TF-IDF + structured + VADER sentiment),
3. defines the **viral** target (top ~11% by total engagement — an imbalanced problem),
4. grid-tunes and trains **10 baseline classifiers + a proposed Viral Stacking Ensemble (VSE)**
   through a leakage-free RF-selection → SMOTE-Tomek pipeline, and
5. reproduces every results table (Tables III–VIII) and the report figures.

> Runtime: ~15–25 min on a free Colab CPU instance (the dataset is 380 MB and the tuning cell is heavy).
> `Runtime → Run all`. All random seeds are fixed at 42.


## 1. Install dependencies

In [ ]:
!pip -q install vaderSentiment imbalanced-learn xgboost >/dev/null
print("dependencies ready")

## 2. Download the dataset from Harvard Dataverse
The file is served by the Dataverse access API; no login required.

In [ ]:
import os, urllib.request
URL = "https://dataverse.harvard.edu/api/access/datafile/11812857"  # tweets_ai.csv
PATH = "tweets_ai.csv"
if not os.path.exists(PATH):
    print("downloading ~380 MB ...")
    urllib.request.urlretrieve(URL, PATH)
print("size (MB):", round(os.path.getsize(PATH)/1e6, 1))

## 3. Preprocessing + feature engineering (VADER sentiment)

In [ ]:
import re, numpy as np, pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
RNG = 42

use = ["id","date","time","tweet","language","urls","photos",
       "replies_count","retweets_count","likes_count","hashtags","video"]
df = pd.read_csv(PATH, usecols=use, dtype={"id":str,"video":str}, low_memory=False)
df = df[df.language == "en"].drop_duplicates("id")
df = df[df.tweet.notna() & (df.tweet.str.strip() != "")]
for c in ["replies_count","retweets_count","likes_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
df["engagement_total"] = df.likes_count + df.retweets_count + df.replies_count

dt = pd.to_datetime(df.date, errors="coerce"); df = df[dt.notna()]; dt = dt[dt.notna()]
df["hour"] = pd.to_numeric(df.time.str[:2], errors="coerce").fillna(12).astype(int)
df["dayofweek"] = dt.dt.dayofweek
df["is_weekend"] = (df.dayofweek >= 5).astype(int)

def n_items(s):
    return 0 if not isinstance(s,str) or s in ("[]","","NA") else s.count(",")+1
df["text_len"] = df.tweet.str.len()
df["word_count"] = df.tweet.str.split().str.len()
df["hashtag_count"] = df.hashtags.apply(n_items)
df["mention_count"] = df.tweet.str.count("@")
df["has_url"] = df.urls.apply(lambda s: 1 if n_items(s)>0 else 0)
df["has_photo"] = df.photos.apply(lambda s: 1 if n_items(s)>0 else 0)
df["has_video"] = (df.video == "1").astype(int)
df["exclam_count"] = df.tweet.str.count("!")
df["question_mark"] = (df.tweet.str.count(r"\?")>0).astype(int)

url_re, men_re = re.compile(r"https?://\S+"), re.compile(r"@\w+")
def clean(t):
    t = url_re.sub(" ", t); t = men_re.sub(" ", t)
    return re.sub(r"\s+"," ",t).strip().lower()
df["clean_text"] = df.tweet.apply(clean)
df = df[df.clean_text.str.len() > 0]

an = SentimentIntensityAnalyzer()
df["vader_compound"] = [an.polarity_scores(t)["compound"] for t in df.clean_text]
df["sentiment"] = np.select([df.vader_compound>=0.05, df.vader_compound<=-0.05],
                            ["positive","negative"], default="neutral")
print("clean tweets:", len(df))

## 4. Define the viral target and build the modelling sample
`viral = 1` if a tweet's total engagement (likes+retweets+replies) is in the top ~11%.

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
VIRAL_THRESHOLD, MODEL_SAMPLE, TFIDF_FEATURES, SELECT_K = 6, 12000, 4000, 300
NUM = ["text_len","word_count","hashtag_count","mention_count","has_url","has_photo",
       "has_video","exclam_count","question_mark","hour","dayofweek","is_weekend","vader_compound"]

df["viral"] = (df.engagement_total >= VIRAL_THRESHOLD).astype(int)
print("overall viral rate:", round(df.viral.mean(), 4))
samp = df.groupby("viral", group_keys=False).sample(frac=MODEL_SAMPLE/len(df), random_state=RNG)
samp = samp.sample(frac=1, random_state=RNG).reset_index(drop=True)

vec = TfidfVectorizer(max_features=TFIDF_FEATURES, ngram_range=(1,2), min_df=5, stop_words="english")
Xtext = vec.fit_transform(samp.clean_text.fillna(""))
X = hstack([Xtext, csr_matrix(samp[NUM].values.astype(float))]).tocsr()
names = np.array(list(vec.get_feature_names_out()) + NUM)
y = samp.viral.values
print("feature matrix:", X.shape, "| sample viral rate:", round(y.mean(),4))

## 5. Models, metrics and the leakage-free pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, cohen_kappa_score,
    confusion_matrix, roc_curve, precision_recall_curve)
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
PROPOSED = "Proposed VSE"
BASELINES = ["Logistic Regression","Naive Bayes","k-NN","Decision Tree","Random Forest",
             "Extra Trees","AdaBoost","SVM (RBF)","XGBoost","MLP (Neural Net)"]

def make_model(name, p=None):
    p = p or {}
    if name=="Logistic Regression": return LogisticRegression(max_iter=1000, random_state=RNG, **p)
    if name=="Naive Bayes": return GaussianNB(**p)
    if name=="k-NN": return KNeighborsClassifier(n_jobs=-1, **p)
    if name=="Decision Tree": return DecisionTreeClassifier(random_state=RNG, **p)
    if name=="Random Forest": return RandomForestClassifier(n_jobs=-1, random_state=RNG, **p)
    if name=="Extra Trees": return ExtraTreesClassifier(n_jobs=-1, random_state=RNG, **p)
    if name=="AdaBoost": return AdaBoostClassifier(random_state=RNG, **p)
    if name=="SVM (RBF)": return SVC(kernel="rbf", random_state=RNG, **p)
    if name=="XGBoost": return XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=RNG, **p)
    if name=="MLP (Neural Net)": return MLPClassifier(max_iter=250, early_stopping=True, random_state=RNG, **p)

GRIDS = {
  "Logistic Regression": {"C":[0.1,1.0,10.0], "class_weight":[None,"balanced"]},
  "Naive Bayes": {"var_smoothing":[1e-9,1e-8,1e-7]},
  "k-NN": {"n_neighbors":[11,15,25], "weights":["uniform","distance"]},
  "Decision Tree": {"max_depth":[10,20,None], "min_samples_leaf":[2,5]},
  "Random Forest": {"n_estimators":[200,400], "max_depth":[None,20], "min_samples_leaf":[1,2]},
  "Extra Trees": {"n_estimators":[200,400], "min_samples_leaf":[1,2]},
  "AdaBoost": {"n_estimators":[100,200], "learning_rate":[0.5,1.0]},
  "SVM (RBF)": {"C":[1.0,2.0,5.0], "gamma":["scale",0.01]},
  "XGBoost": {"n_estimators":[200,400], "max_depth":[4,6], "learning_rate":[0.05,0.1]},
  "MLP (Neural Net)": {"hidden_layer_sizes":[(64,),(96,)], "alpha":[1e-4,1e-3]}}

def proposed_model(best):
    return StackingClassifier(
      estimators=[("lr", make_model("Logistic Regression", best.get("Logistic Regression"))),
                  ("rf", make_model("Random Forest", best.get("Random Forest"))),
                  ("xgb", make_model("XGBoost", best.get("XGBoost")))],
      final_estimator=LogisticRegression(max_iter=1000, random_state=RNG),
      stack_method="predict_proba", cv=3, n_jobs=-1)

def scores_of(clf, Xte):
    return clf.predict_proba(Xte)[:,1] if hasattr(clf,"predict_proba") else clf.decision_function(Xte)

def metric_row(y, yhat, sc):
    return {"Accuracy":accuracy_score(y,yhat), "Precision":precision_score(y,yhat,zero_division=0),
            "Recall":recall_score(y,yhat), "F1":f1_score(y,yhat),
            "Macro-F1":f1_score(y,yhat,average="macro"), "ROC-AUC":roc_auc_score(y,sc),
            "PR-AUC":average_precision_score(y,sc), "MCC":matthews_corrcoef(y,yhat),
            "Kappa":cohen_kappa_score(y,yhat)}

## 6. Table III — per-algorithm hyperparameter tuning (leakage-free)
Each algorithm is grid-searched with SMOTE-Tomek re-fitted inside every CV fold, so
no synthetic samples leak between folds. (This cell is the slowest — a few minutes.)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)
rank = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(Xtr, ytr)
idx = np.argsort(rank.feature_importances_)[::-1][:SELECT_K]
Xtr_raw, Xte_raw = Xtr[:,idx].toarray(), Xte[:,idx].toarray()
sca = StandardScaler().fit(Xtr_raw)
Xtr_s, Xte_s = sca.transform(Xtr_raw), sca.transform(Xte_raw)
Xtr_r, ytr_r = SMOTETomek(random_state=RNG).fit_resample(Xtr_s, ytr)

best_params, tune_rows = {}, []
for name in BASELINES:
    grid = {f"clf__{k}":v for k,v in GRIDS[name].items()}
    pipe = ImbPipeline([("sc",StandardScaler()),("sm",SMOTETomek(random_state=RNG)),("clf",make_model(name))])
    Xg, yg = Xtr_raw, ytr
    if name=="SVM (RBF)" and len(yg)>5000:
        s=np.random.RandomState(RNG).permutation(len(yg))[:5000]; Xg,yg=Xg[s],yg[s]
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=3, n_jobs=-1).fit(Xg, yg)
    best = {k.replace("clf__",""):v for k,v in gs.best_params_.items()}
    best_params[name] = best
    Xf,yf = Xtr_r, ytr_r
    if name=="SVM (RBF)" and len(yf)>9000:
        s2=np.random.RandomState(RNG).permutation(len(yf))[:9000]; Xf,yf=Xf[s2],yf[s2]
    f1_def = f1_score(yte, make_model(name).fit(Xf,yf).predict(Xte_s))
    f1_tun = f1_score(yte, make_model(name,best).fit(Xf,yf).predict(Xte_s))
    tune_rows.append({"Model":name, "Best configuration":str(best),
                      "F1 (default)":round(f1_def,4), "F1 (tuned)":round(f1_tun,4)})
    print(f"{name:20s} tuned F1={f1_tun:.3f} (def {f1_def:.3f})")
tab_tuning = pd.DataFrame(tune_rows).set_index("Model"); tab_tuning

## 7. Table IV — full hold-out test performance (11 models × 9 metrics)

In [ ]:
rows, scores, preds = [], {"y_true":yte}, {}
for name in BASELINES:
    Xf,yf = Xtr_r, ytr_r
    if name=="SVM (RBF)" and len(yf)>9000:
        s2=np.random.RandomState(RNG).permutation(len(yf))[:9000]; Xf,yf=Xf[s2],yf[s2]
    clf = make_model(name, best_params[name]).fit(Xf, yf)
    yhat, sc = clf.predict(Xte_s), scores_of(clf, Xte_s)
    scores[name], preds[name] = sc, yhat
    rows.append({"Model":name, **metric_row(yte, yhat, sc)})
vse = proposed_model(best_params).fit(Xtr_r, ytr_r)
yhat, sc = vse.predict(Xte_s), scores_of(vse, Xte_s)
scores[PROPOSED], preds[PROPOSED] = sc, yhat
rows.append({"Model":PROPOSED, **metric_row(yte, yhat, sc)})
tab4 = pd.DataFrame(rows).round(4).set_index("Model")
imp = pd.Series(rank.feature_importances_[idx], index=names[idx]).sort_values(ascending=False)
tab4

## 8. Table V — 10-fold cross-validation (mean ± std, tuned models)

In [ ]:
rank2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(X, y)
idx2 = np.argsort(rank2.feature_importances_)[::-1][:SELECT_K]
Xs = X[:,idx2].toarray()
def pipe(clf): return ImbPipeline([("sc",StandardScaler()),("sm",SMOTETomek(random_state=RNG)),("clf",clf)])
skf = StratifiedKFold(10, shuffle=True, random_state=RNG)
chosen = {"Random Forest":make_model("Random Forest",best_params["Random Forest"]),
          "XGBoost":make_model("XGBoost",best_params["XGBoost"]), PROPOSED:proposed_model(best_params)}
fold_F1 = {}; rows=[]
for name,clf in chosen.items():
    p = pipe(clf); f1s, pra, mcs = [], [], []
    for tr,te in skf.split(Xs,y):
        p.fit(Xs[tr],y[tr]); yh=p.predict(Xs[te]); sc=scores_of(p,Xs[te])
        f1s.append(f1_score(y[te],yh)); pra.append(average_precision_score(y[te],sc)); mcs.append(matthews_corrcoef(y[te],yh))
    fold_F1[name]=np.array(f1s)
    rows.append({"Model":name, "F1":f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
                 "PR-AUC":f"{np.mean(pra):.3f} ± {np.std(pra):.3f}", "MCC":f"{np.mean(mcs):.3f} ± {np.std(mcs):.3f}"})
tab5 = pd.DataFrame(rows).set_index("Model"); tab5

## 9. Table VI — paired statistical significance tests

In [ ]:
from scipy import stats
sig=[]
for base in ["Random Forest","XGBoost"]:
    a,b = fold_F1[PROPOSED], fold_F1[base]; diff=a-b
    sig.append({"Comparison":f"{PROPOSED} vs {base} (F1)",
                "t-test p":round(stats.ttest_rel(a,b).pvalue,4),
                "Wilcoxon p":round(stats.wilcoxon(a,b).pvalue,4),
                "Mean diff":round(diff.mean(),4),
                "Cohen d":round(diff.mean()/(diff.std(ddof=1)+1e-12),3),
                "Significant":"Yes" if min(stats.ttest_rel(a,b).pvalue, stats.wilcoxon(a,b).pvalue)<0.05 else "No"})
pd.DataFrame(sig).set_index("Comparison")

## 10. Table VII — five-stage incremental ablation (RF probe; S4 = full VSE)

In [ ]:
def evalt(Xa_tr, Xa_te, clf, resample):
    sc = StandardScaler(with_mean=False); a=sc.fit_transform(Xa_tr); b=sc.transform(Xa_te); yy=ytr
    if resample: a, yy = SMOTETomek(random_state=RNG).fit_resample(a, ytr)
    clf.fit(a, yy); yh=clf.predict(b)
    return {"Accuracy":accuracy_score(yte,yh),"Recall":recall_score(yte,yh),
            "F1":f1_score(yte,yh),"MCC":matthews_corrcoef(yte,yh)}
rf=lambda: make_model("Random Forest", best_params["Random Forest"])
nt=TFIDF_FEATURES
ab=[("S0","TF-IDF text only + RF",evalt(Xtr[:,:nt],Xte[:,:nt],rf(),False)),
    ("S1","+ structured features",evalt(Xtr,Xte,rf(),False)),
    ("S2","+ RF feature selection",evalt(Xtr[:,idx].toarray(),Xte[:,idx].toarray(),rf(),False)),
    ("S3","+ SMOTE-Tomek",evalt(Xtr[:,idx].toarray(),Xte[:,idx].toarray(),rf(),True)),
    ("S4","+ Stacking (full VSE)",evalt(Xtr[:,idx].toarray(),Xte[:,idx].toarray(),proposed_model(best_params),True))]
b0=ab[0][2]["F1"]
tab7=pd.DataFrame([{"Stage":s,"Description":d,**{k:round(v,4) for k,v in m.items()},
                   "dF1 vs S0":f"{(m['F1']-b0)*100:+.2f} pp"} for s,d,m in ab]).set_index("Stage"); tab7

## 11. Key figures (ROC, precision–recall, confusion matrix)

In [ ]:
import matplotlib.pyplot as plt
order = BASELINES + [PROPOSED]
yt = scores["y_true"]
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
for name in order:
    fpr,tpr,_ = roc_curve(yt, scores[name]); lw = 2.6 if name==PROPOSED else 1.2
    ax[0].plot(fpr,tpr,lw=lw,label=f"{name} ({roc_auc_score(yt,scores[name]):.3f})")
ax[0].plot([0,1],[0,1],"--",color="grey"); ax[0].set_title("ROC curves")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].legend(fontsize=6.5)
for name in order:
    pr,rc,_ = precision_recall_curve(yt, scores[name]); lw = 2.6 if name==PROPOSED else 1.2
    ax[1].plot(rc,pr,lw=lw,label=f"{name} ({average_precision_score(yt,scores[name]):.3f})")
ax[1].set_title("Precision-Recall curves"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend(fontsize=6.5)
plt.tight_layout(); plt.show()

cm = confusion_matrix(yt, preds[PROPOSED])
print("Proposed VSE confusion matrix (rows=actual, cols=pred):\n", cm)
print("\nTop-15 RF-ranked features:\n", imp.head(15))

## 12. Table VIII — comparison of the proposed model with existing studies
The proposed VSE is positioned against the modelling studies from the report's
literature review. **Read with care:** those studies address different tasks
(sentiment, engagement tiers, retweet counts) on different datasets with different
class balance, and most report accuracy on comparatively balanced problems, whereas
this study is imbalanced binary virality where F1 / PR-AUC / MCC are the honest
metrics. The table positions this study, it is not a like-for-like benchmark.

In [ ]:
comparison = pd.DataFrame([
 ["Rustam et al. [4]","Extra Trees / XGBoost / LSTM","7,528 COVID-19 tweets (sentiment)","Acc 0.93 (extra trees)"],
 ["Chakraborty et al. [5]","Fuzzy rule base + classical","226,668 COVID-19 tweets (sentiment)","Acc up to 0.81"],
 ["Naseem et al. [6]","Classical vs deep vs transformer","90,000 tweets (COVIDSenti, sentiment)","Transformer models best"],
 ["Vohra & Garg [7]","CNN + FastText","358,823 WFH tweets (sentiment)","Acc 0.926"],
 ["Alharbi & de Doncker [9]","CNN + user behaviour","SemEval Twitter (sentiment)","Beats text-only baselines"],
 ["Andariesta & Wasesa [10]","LR / DT / KNN / RF","12,786 e-commerce tweets (engagement)","4-tier engagement classification"],
 ["Meštrović et al. [11]","BERT + multilayer network","199,431 tweets (retweet-count classes)","Text + network best"],
 ["Proposed VSE (this study)","Resampled stacking ensemble","893,076 AI tweets (virality, imbalanced)",
  f"F1 {tab4.loc[PROPOSED,'F1']:.3f}, PR-AUC {tab4.loc[PROPOSED,'PR-AUC']:.3f}, MCC {tab4.loc[PROPOSED,'MCC']:.3f}"],
], columns=["Existing study","Model","Dataset (task)","Reported performance"]).set_index("Existing study")
comparison

## 13. Notes
- All tables above (III–VIII) and the figures are computed live from the downloaded data; nothing is hard-coded.
- Numbers should match the report closely; tiny differences can occur if Colab ships different library versions.
- To host on GitHub and open in Colab: push this `.ipynb` to a repo, then use
  `https://colab.research.google.com/github/<user>/<repo>/blob/main/SWA2124_Viral_Analysis.ipynb`.
- **Dataset citation:** N. de Marcellis-Warin, D. Kouloukoui, and T. Warin, "A large-scale dataset of
  AI-related tweets: Structure and descriptive statistics," *Data in Brief*, vol. 62, art. 111960, 2025,
  Harvard Dataverse, DOI: 10.7910/DVN/NHLEJL.
